In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import pickle
import time
import sys
import os
sys.path.append('../')
from helpers import *
from fns_grid import *

 
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
 
def get_rays(H, W, focal, c2w):
    """Generate ray origins and directions for a camera pose."""
    i, j = np.meshgrid(np.arange(W, dtype=np.float32),
                       np.arange(H, dtype=np.float32), indexing='xy')
    dirs = np.stack([(i - W * 0.5) / focal,
                     -(j - H * 0.5) / focal,
                     -np.ones_like(i)], -1)
    rays_d = np.sum(dirs[..., np.newaxis, :] * c2w[:3, :3], -1)
    rays_o = np.broadcast_to(c2w[:3, -1], rays_d.shape).copy()
    return rays_o, rays_d
 
 
def volume_render(raw, z_vals):
    rgb = torch.sigmoid(raw[..., :3])
    sigma_a = torch.relu(raw[..., 3])
    dists = torch.cat([z_vals[..., 1:] - z_vals[..., :-1],
                       torch.full_like(z_vals[..., :1], 1e10)], -1)
    alpha = 1.0 - torch.exp(-sigma_a * dists)
    trans = torch.clamp(1.0 - alpha + 1e-10, max=1.0)
    trans = torch.cat([torch.ones_like(trans[..., :1]), trans[..., :-1]], -1)
    weights = alpha * torch.cumprod(trans, -1)
    rgb_map = torch.sum(weights[..., None] * rgb, -2)
    depth_map = torch.sum(weights * z_vals, -1)
    acc_map = torch.sum(weights, -1)
    return rgb_map, depth_map, acc_map
 
 
def normalize_pts(pts, near, far):
    scene_range = far  # works for forward-facing / 360° object-centric scenes
    return (pts / scene_range) * 0.5 + 0.5  # map [-far, far] -> [0, 1]

filename = 'lego_400.npz'
if not os.path.exists(filename):
    !gdown --id 108jNfjPITTsTA0lE6Kpg7Ei53BUVL-4n # Lego

data = np.load(filename)
images = data['images']
poses = data['poses']
focal = data['focal']
H, W = images.shape[1:3]

images, val_images, test_images = np.split(images[...,:3], [100,107], axis=0)
poses, val_poses, test_poses = np.split(poses, [100,107], axis=0)

TEST_IDX = 10

print(val_images.shape, test_images.shape, focal)
plt.imshow(test_images[TEST_IDX,...])
plt.show()

In [ ]:
def sample_voxel_grid(grid, pts, bbox_min, bbox_max):
    # Normalise to [-1, 1] for grid_sample (DHW order → zyx)
    normalised = 2.0 * (pts - bbox_min) / (bbox_max - bbox_min) - 1.0
 
    # grid_sample expects (N, C, D_in, H_in, W_in) and
    # sample grid of shape  (N, D_out, H_out, W_out, 3) with last dim = (x,y,z)
    # We have a single 'output pixel' per query point so D_out=H_out=W_out=1
    sample_coords = normalised[:, [2, 1, 0]]          # (N,3) reorder xyz → whd for grid_sample's (x,y,z)
    sample_coords = sample_coords.view(1, 1, 1, -1, 3)  # (1,1,1,N,3)
 
    sampled = F.grid_sample(grid, sample_coords,
                            mode='bilinear', padding_mode='zeros',
                            align_corners=True)          # (1,C,1,1,N)
    return sampled.view(grid.shape[1], -1).permute(1, 0)  # (N, C)
 
 
def fit_grid_nerf(images, poses, focal, H, W,
                  val_images=None, val_poses=None,
                  r=128, iters=50000, lr=1e-2,
                  batch_size=1024, N_samples=128,
                  near=2., far=6., stratified=True,
                  bbox_min=None, bbox_max=None,
                  tv_weight=0.0,
                  log_interval=5000, seed=0, device=device,
                  count_params=False):
    torch.manual_seed(seed)
    np.random.seed(seed)
 
    # ------------------------------------------------------------------
    # Pre-compute and shuffle training rays from images + poses
    # ------------------------------------------------------------------
    all_rays_o_list, all_rays_d_list, all_rgb_list = [], [], []
    for i in range(images.shape[0]):
        ro, rd = get_rays(H, W, focal, poses[i])
        all_rays_o_list.append(ro.reshape(-1, 3))
        all_rays_d_list.append(rd.reshape(-1, 3))
        all_rgb_list.append(images[i][..., :3].reshape(-1, 3))
    all_rays_o = np.concatenate(all_rays_o_list, 0)
    all_rays_d = np.concatenate(all_rays_d_list, 0)
    all_rgb    = np.concatenate(all_rgb_list, 0)
 
    perm = np.random.permutation(all_rays_o.shape[0])
    all_rays_o = all_rays_o[perm]
    all_rays_d = all_rays_d[perm]
    all_rgb    = all_rgb[perm]
 
    # Default bounding box: cube that covers the lego-style scene
    if bbox_min is None:
        bbox_min = np.array([-1.5, -1.5, -1.5], dtype=np.float32)
    if bbox_max is None:
        bbox_max = np.array([ 1.5,  1.5,  1.5], dtype=np.float32)
    bbox_min_t = torch.tensor(bbox_min, dtype=torch.float32, device=device)
    bbox_max_t = torch.tensor(bbox_max, dtype=torch.float32, device=device)
 
    # Learnable voxel grid: 4 channels (rgb logits + sigma), resolution r^3
    grid = torch.zeros(1, 4, r, r, r, device=device, requires_grad=True)
    nn.init.uniform_(grid.data, -0.1, 0.1)
    num_params = r ** 3 * 4
    if count_params:
        print(f'Number of parameters: {num_params}')
 
    optimizer = optim.Adam([grid], lr=lr)
    losses, xs = [], []
    best_loss = float('inf')
    b_i = 0
    t0 = time.time()
 
    for it in range(iters):
        # Cycle through pre-shuffled rays
        if b_i + batch_size > all_rays_o.shape[0]:
            b_i = 0
        ro = torch.tensor(all_rays_o[b_i:b_i + batch_size],
                          dtype=torch.float32, device=device)
        rd = torch.tensor(all_rays_d[b_i:b_i + batch_size],
                          dtype=torch.float32, device=device)
        target = torch.tensor(all_rgb[b_i:b_i + batch_size],
                              dtype=torch.float32, device=device)
        b_i += batch_size
 
        # Depth samples
        z_vals = torch.linspace(near, far, N_samples, device=device)
        if stratified:
            z_vals = z_vals + torch.rand(batch_size, N_samples, device=device) * (far - near) / N_samples
        else:
            z_vals = z_vals.unsqueeze(0).expand(batch_size, -1)
 
        # 3-D sample points: (batch, N_samples, 3)
        pts = ro[:, None, :] + rd[:, None, :] * z_vals[:, :, None]
 
        # Query grid
        raw = sample_voxel_grid(grid, pts.reshape(-1, 3),
                                bbox_min_t, bbox_max_t)   # (batch*N_samples, 4)
        raw = raw.reshape(batch_size, N_samples, 4)
 
        # Volume render
        rgb_map, _, _ = volume_render(raw, z_vals)
 
        loss = torch.mean((rgb_map - target) ** 2)
 
        # Optional total-variation regularisation for smoother grids
        if tv_weight > 0:
            tv = (torch.diff(grid, dim=2).pow(2).mean() +
                  torch.diff(grid, dim=3).pow(2).mean() +
                  torch.diff(grid, dim=4).pow(2).mean())
            loss = loss + tv_weight * tv
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        losses.append(loss.item())
        xs.append(it)
        if loss.item() < best_loss:
            best_loss = loss.item()
 
        if (it + 1) % log_interval == 0:
            psnr = -10.0 * np.log10(loss.item())
            elapsed = (time.time() - t0) / 60.
            print(f'Iteration {it+1}/{iters}, Loss: {loss.item():.3e}, '
                  f'PSNR: {psnr:.2f} dB, Time: {elapsed:.1f} min')
 
    # ------------------------------------------------------------------
    # Validation render (renders first val image if val data provided)
    # ------------------------------------------------------------------
    val_psnr = None
    pred_image = None
    if val_images is not None and val_poses is not None:
        val_img = val_images[0]
        ro_v, rd_v = get_rays(H, W, focal, val_poses[0])
        ro_flat = ro_v.reshape(-1, 3)
        rd_flat = rd_v.reshape(-1, 3)
        rgb_chunks = []
        chunk = 512
        with torch.no_grad():
            for ci in range(0, ro_flat.shape[0], chunk):
                ro_c = torch.tensor(ro_flat[ci:ci + chunk],
                                    dtype=torch.float32, device=device)
                rd_c = torch.tensor(rd_flat[ci:ci + chunk],
                                    dtype=torch.float32, device=device)
                n = ro_c.shape[0]
                z = torch.linspace(near, far, N_samples, device=device) \
                         .unsqueeze(0).expand(n, -1)
                pts_c = ro_c[:, None, :] + rd_c[:, None, :] * z[:, :, None]
                raw_c = sample_voxel_grid(grid, pts_c.reshape(-1, 3),
                                          bbox_min_t, bbox_max_t)
                raw_c = raw_c.reshape(n, N_samples, 4)
                rgb_c, _, _ = volume_render(raw_c, z)
                rgb_chunks.append(rgb_c.cpu().numpy())
        pred_image = np.concatenate(rgb_chunks, 0).reshape(H, W, 3)
        val_loss = np.mean((pred_image - val_img[..., :3]) ** 2)
        val_psnr = -10.0 * np.log10(val_loss)
        print(f'Val PSNR: {val_psnr:.2f} dB')
 
    return {
        'grid': grid.detach(),
        'pred': pred_image,
        'losses': losses,
        'xs': xs,
        'best_loss': best_loss,
        'val_psnr': val_psnr,
    }
 
 
def render_image_from_grid(grid, pose, H, W, focal,
                           near=2., far=6., N_samples=128,
                           bbox_min=None, bbox_max=None,
                           chunk=512, device=device):
    if bbox_min is None:
        bbox_min = np.array([-1.5, -1.5, -1.5], dtype=np.float32)
    if bbox_max is None:
        bbox_max = np.array([ 1.5,  1.5,  1.5], dtype=np.float32)
    bbox_min_t = torch.tensor(bbox_min, dtype=torch.float32, device=device)
    bbox_max_t = torch.tensor(bbox_max, dtype=torch.float32, device=device)
 
    rays_o, rays_d = get_rays(H, W, focal, pose)
    ro_flat = rays_o.reshape(-1, 3)
    rd_flat = rays_d.reshape(-1, 3)
 
    rgb_chunks, depth_chunks = [], []
    with torch.no_grad():
        for ci in range(0, ro_flat.shape[0], chunk):
            ro_c = torch.tensor(ro_flat[ci:ci + chunk],
                                dtype=torch.float32, device=device)
            rd_c = torch.tensor(rd_flat[ci:ci + chunk],
                                dtype=torch.float32, device=device)
            n = ro_c.shape[0]
            z = torch.linspace(near, far, N_samples, device=device) \
                     .unsqueeze(0).expand(n, -1)
            pts = ro_c[:, None, :] + rd_c[:, None, :] * z[:, :, None]
            raw = sample_voxel_grid(grid, pts.reshape(-1, 3),
                                    bbox_min_t, bbox_max_t)
            raw = raw.reshape(n, N_samples, 4)
            rgb_c, depth_c, _ = volume_render(raw, z)
            rgb_chunks.append(rgb_c.cpu().numpy())
            depth_chunks.append(depth_c.cpu().numpy())
 
    rgb_image = np.concatenate(rgb_chunks, 0).reshape(H, W, 3)
    depth_map = np.concatenate(depth_chunks, 0).reshape(H, W)
    return rgb_image, depth_map

In [ ]:
N_samples = 512
batch_size = 1024 #2^14
num_layers, num_channels = 2, 128
near, far = 2., 6.

model_sizes = [41**3, 76**3]
outputs = {}
for model_size in model_sizes:
    r = compute_params_from_model_size("grid_eta", None, model_size, n_dims=3)
    print(f'grid_reso: {r}')
    output = fit_grid_nerf(
        images, poses, focal, H, W,
        val_images=val_images, val_poses=val_poses,
        r=r, iters=10000, lr=5e-2,
        near=2., far=6., N_samples=128,
        tv_weight=1e-5, log_interval=1000, device=device, seed=0, count_params=True
    )
    outputs[f'{r}'] = output

In [ ]:
outputs_plot = {}
for param_val in outputs.keys():
    rendered, _ = render_image_from_grid(
        outputs[param_val]['grid'], test_poses[TEST_IDX], H, W, focal,
        near=near, far=far, N_samples=N_samples, device=device
    )
    outputs_plot[param_val] = {}
    outputs_plot[param_val]['best_pred'] = rendered

with open(f"3d_nerf/grid.pkl", "wb") as f:
    pickle.dump(outputs_plot, f)
plot_error_heatmaps(test_images[TEST_IDX,...], outputs_plot, model_name="Grid")